# Wrist reorientation and burst-evoked HR
For bursts of similar AUC, is the HR response different when movement ends in a **new stable wrist gravity direction**?

This is an exploratory within-recording analysis, **not a sleeping-position classifier**. Gravity reveals tilt, not heading: rotations about gravity can be invisible, and the wrist can move independently of the trunk. "Little reorientation" does not mean no movement. "Sustained" refers only to the two measured post-burst windows.

Run all cells in a Python environment with the notebook requirements installed. This notebook loads the FIT itself and does not depend on `load_fit_test.ipynb` having run. Change `FIT_PATH` below for another recording.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
PROCESSING = HERE if (HERE / "wrist_reorientation.py").exists() else HERE / "offline_processing"
if not (PROCESSING / "wrist_reorientation.py").exists():
    raise FileNotFoundError("Open this notebook from the repository or offline_processing folder.")
if str(PROCESSING) not in sys.path:
    sys.path.insert(0, str(PROCESSING))

from fit_recording import load_recording
from detect_acc_bursts import prepare_acceleration, detect_bursts
from burst_hr_response import bbi_to_hr, analyze_burst_hr, TERTILES
from wrist_reorientation import (
    prepare_wrist_axes, measure_reorientation, classify_reorientation,
    angle_degrees, LITTLE, CHANGED, INTERMEDIATE, UNKNOWN, CLASSES,
)
from reorientation_hr import join_hr_orientation, summarize_groups, match_by_auc, summarize_pairs

from notebook_paths import input_path

FIT_PATH = input_path("reorientation_fit_path")
INPUT_UNIT = "mg"  # This recording has numeric milli-g values despite FIT labels saying g.
FS = 100.0
BURST_THRESHOLD_G = 0.020
MERGE_GAP_S = 5.0
HR_MAX_GAP_S = 3.0
ISOLATION_S = 30.0
EXCLUDE_LATE_OVERLAP = True
ARTIFACTS = []  # Optional reviewed (start_UTC, end_UTC) intervals; empty means unreviewed.
ORIENTATION = dict(
    sampling_rate=FS, window_s=10.0, buffer_s=2.0, hold_s=10.0,
    min_coverage=0.95, max_gap_s=0.25, max_rms_g=0.05, max_spread_deg=10.0,
    gravity_limits_g=(0.8, 1.2), little_angle_deg=10.0,
    change_angle_deg=30.0, persistence_angle_deg=10.0,
)
MAX_AUC_RATIO = 1.25
EXPORT = False

COLORS = {LITTLE: "#247b9b", CHANGED: "#b43c69", INTERMEDIATE: "#7c6c24", UNKNOWN: "#858585"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


## Method and interpretation
- **Movement:** reuse the existing magnitude-based detector with a **20 mg** threshold borrowed from the reference study; it is not validated for this Garmin pipeline. Keep its current 5 s merge rule. AUC is in g*s; its tertiles are calculated over **all candidate bursts**, before exclusions.
- **Orientation:** preserve raw XYZ and gravity; do not use the magnitude or high-pass-filtered signal. Estimate gravity direction from component medians in pre `[start-12, start-2)`, post `[end+2, end+12)`, and hold `[end+12, end+22)` seconds.
- **Window QC:** require >=95% samples, no gap >0.25 s, median-vector norm 0.8-1.2 g, vector residual RMS <=0.05 g, and 95th-percentile angular deviation <=10 degrees. Reject overlap with another detected burst.
- **Classes:** both post windows must be within 10 degrees of one another. Relative to pre, both <=10 degrees means little reorientation; both >=30 degrees means sustained reorientation. Intermediate angles remain separate; failed QC or persistence is unknown.
- **HR:** reuse the shared [paper-inspired HR analysis](https://www.nature.com/articles/s41598-025-29723-7): the current implementation uses -19 to +54 s at 1 Hz and a 14-s baseline [-19,-5), with 30 s clear isolation on both sides. The longer epoch and baseline are local adaptations. Here, additionally exclude another movement anywhere up to +54 s by default. This is stricter than the earlier notebook's default.
- **Similar AUC:** compare within the original tertiles, then make one-to-one pairs within each tertile with max(AUC)/min(AUC) <=1.25. Matching maximizes pair count then minimizes log-AUC distance; it never uses HR outcomes.

**The 20 mg movement threshold was validated in the reference study, not on this Garmin pipeline.** Orientation windows, stability limits, angle cutoffs, and the matching caliper are exploratory. The paper's torso-position criteria cannot validate wrist orientation. AUC stratification and matching are extensions, not an exact replication of the paper.

BBI times are callback arrival estimates, not exact beat timestamps. Gaps, invalid intervals, and incomplete HR epochs are excluded; no missing beats are invented. No raw ECG/PPG morphology is available for artifact validation. Matching AUC alone does not control duration, time of night, sleep stage, or artifacts. Results and event-level SEM are descriptive for this recording, not population-level inference.


In [ ]:
print(f"Loading {FIT_PATH.name}; a full overnight FIT can take a few minutes.")
recording = load_recording(FIT_PATH)
accel_df = recording["accel_df"]
observations, bbi_quality = bbi_to_hr(recording["bbi_rows"])
display(pd.Series({
    "XYZ samples": len(accel_df),
    "Start (UTC)": accel_df.sample_time.min(),
    "End (UTC)": accel_df.sample_time.max(),
    "Decoded BBI intervals": len(recording["bbi_rows"]),
    "HR callbacks": len(observations),
    "Numerical input unit (explicit)": INPUT_UNIT,
}, name="Recording"))
display(recording["fit_field_units"])
display(recording["bbi_report"])
display(bbi_quality)


In [ ]:
magnitude_g, acc_quality = prepare_acceleration(
    accel_df, sampling_rate=FS, input_unit=INPUT_UNIT, max_gap_s=0.25,
)
median_g = float(magnitude_g.median())
if not 0.5 <= median_g <= 1.5:
    raise ValueError(f"Median magnitude is {median_g:.3f} g. Check INPUT_UNIT before proceeding.")
bursts, burst_signals = detect_bursts(
    magnitude_g, sampling_rate=FS, alfa=BURST_THRESHOLD_G,
    merge_gap_s=MERGE_GAP_S, return_signals=True,
)
hr = analyze_burst_hr(
    bursts, observations, accel_df.sample_time.min(), accel_df.sample_time.max(),
    max_gap_s=HR_MAX_GAP_S, artifacts=ARTIFACTS, isolation_s=ISOLATION_S,
    exclude_late_overlap=EXCLUDE_LATE_OVERLAP,
)
display(pd.Series(acc_quality, name="Acceleration QC"))
display(hr["report"])
print(f"Detected {len(bursts)} bursts; {hr['events'].included.sum()} pass HR and isolation criteria.")


## Orientation measurements and exclusions
The orientation script uses timestamped XYZ directly, without filling gaps. Other bursts are checked against **all** candidates, including those with unusable HR. Inspect exclusions before looking at group differences.


In [ ]:
axes_g = prepare_wrist_axes(accel_df, input_unit=INPUT_UNIT)
orientation = measure_reorientation(axes_g, hr["events"], **ORIENTATION)
events = join_hr_orientation(hr["events"], orientation)
group_summary, group_curves = summarize_groups(events, hr["epochs_pct"])

display(pd.Series({
    "Candidate bursts": len(events),
    "HR eligible": int(events.included.sum()),
    "Stable orientation windows": int(events.window_qc_pass.sum()),
    "Stable and persistent orientation": int(events.orientation_usable.sum()),
    "HR + little/sustained orientation eligible": int(events.comparison_included.sum()),
}, name="Selection"))
display(pd.crosstab(events.auc_tertile, events.reorientation_class, dropna=False).reindex(
    index=TERTILES, columns=CLASSES, fill_value=0))
display(group_summary)
excluded = events.loc[~events.comparison_included, "comparison_exclusion_reason"]
display(excluded.value_counts().rename("bursts").to_frame())
display(axes_g.attrs)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
origin = accel_df.sample_time.min()
for label in CLASSES:
    selected = events.loc[events.reorientation_class == label]
    hours = (selected.start - origin).dt.total_seconds() / 3600
    ax.scatter(hours, selected.tilt_change_deg, s=18, alpha=0.7,
               color=COLORS[label], label=f"{label} (n={len(selected)})")
ax.axhline(ORIENTATION["little_angle_deg"], color=COLORS[LITTLE], linestyle=":", linewidth=1)
ax.axhline(ORIENTATION["change_angle_deg"], color=COLORS[CHANGED], linestyle=":", linewidth=1)
ax.set(xlabel="Hours since recording start", ylabel="Wrist tilt change (degrees)",
       ylim=(-5, 185), title="All detected bursts; undefined angles are not plotted")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()


## HR response within AUC tertiles
Curves are percentage change from each burst's baseline. Shading is +/-1 SEM **across events in this recording**, not across participants. These groups are not yet pair-matched; within-tertile AUC distributions may still differ.


In [ ]:
def plot_hr_groups(curves, series_column, paired=False):
    rows = 2 if paired else 1
    fig, panels = plt.subplots(rows, 3, figsize=(13, 3.4 * rows), sharex=True, sharey='row', squeeze=False)
    for j, tertile in enumerate(TERTILES):
        ax = panels[0, j]
        for label in [LITTLE, CHANGED]:
            curve = curves.loc[(curves.auc_tertile == tertile) & (curves[series_column] == label)]
            n = int(curve.n.iloc[0]) if len(curve) else 0
            if n:
                x, mean, sem = (curve[key].to_numpy(dtype=float) for key in ["relative_s", "mean_pct", "sem_pct"])
                short_label = 'Little' if label == LITTLE else 'Sustained'
                ax.plot(x, mean, color=COLORS[label], label=f"{short_label} (n={n})")
                if n > 1:
                    ax.fill_between(x, mean - sem, mean + sem, color=COLORS[label], alpha=0.15)
        if ax.lines:
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, "No eligible events", ha="center", transform=ax.transAxes)
        ax.set(title=f"{tertile} AUC", ylabel="HR change (%)")
        if paired:
            ax = panels[1, j]
            curve = curves.loc[(curves.auc_tertile == tertile) & (curves[series_column] == "Paired difference")]
            n = int(curve.n.iloc[0]) if len(curve) else 0
            if n:
                x, mean, sem = (curve[key].to_numpy(dtype=float) for key in ["relative_s", "mean_pct", "sem_pct"])
                ax.plot(x, mean, color="#436b46")
                if n > 1:
                    ax.fill_between(x, mean - sem, mean + sem, color="#436b46", alpha=0.15)
            else:
                ax.text(0.5, 0.5, "No matched pairs", ha="center", transform=ax.transAxes)
            ax.set(ylabel="Paired difference (pp)", title=f"Sustained minus little; pairs = {n}")
        for ax in panels[:, j]:
            ax.axhline(0, color="0.7", linewidth=0.8)
            ax.axvline(0, color="0.4", linestyle=":", linewidth=0.8)
            ax.axvspan(-19, -5, color="0.5", alpha=0.07)
            ax.set_xlim(-19, 40)
        panels[-1, j].set_xlabel("Seconds from burst onset")
    fig.tight_layout()
    plt.show()
    return fig

tertile_figure = plot_hr_groups(group_curves, "reorientation_class")


## One-to-one similar-AUC comparisons
Pair selection uses only eligibility, orientation class, AUC tertile, and AUC distance. Each burst appears in at most one pair. Positive peak increase is measured at +1 through +54 s and clipped to zero when HR never rises; signed post-epoch mean change is also retained. Differences between HR percentages are **percentage points (pp)**.

Small or empty matched sets are a result, not evidence of no physiological effect. Inspect AUC balance, duration, and time separation in the pair table.


In [ ]:
pairs = match_by_auc(events, max_auc_ratio=MAX_AUC_RATIO)
pair_summary, pair_curves = summarize_pairs(pairs, hr["epochs_pct"])
print(f"Matched pairs: {len(pairs)}; comparison-eligible bursts: {events.comparison_included.sum()}.")
if len(pairs) < 2:
    print('Pair-level SEM is undefined with fewer than two pairs. Do not infer a group effect from this recording.')
display(pair_summary)
display(pairs)
matched_figure = plot_hr_groups(pair_curves, "series", paired=True)


## Angle-threshold sensitivity
Hold the little-change limit, stability checks, and AUC caliper fixed while varying the sustained-change limit. This reuses the same raw measurements and original AUC tertiles. Report the full grid; do not select a threshold because it gives the largest HR difference.


In [ ]:
sensitivity_rows = []
for threshold in [20.0, 30.0, 45.0]:
    classified = classify_reorientation(
        orientation, little_angle_deg=ORIENTATION["little_angle_deg"],
        change_angle_deg=threshold, persistence_angle_deg=ORIENTATION["persistence_angle_deg"],
    )
    compared = join_hr_orientation(hr["events"], classified)
    matched = match_by_auc(compared, max_auc_ratio=MAX_AUC_RATIO)
    for tertile in TERTILES:
        selected = compared.loc[(compared.auc_tertile == tertile) & compared.comparison_included]
        matched_group = matched.loc[matched.auc_tertile == tertile]
        sensitivity_rows.append({
            "change_threshold_deg": threshold, "auc_tertile": tertile,
            "little_n": int((selected.reorientation_class == LITTLE).sum()),
            "sustained_n": int((selected.reorientation_class == CHANGED).sum()),
            "pairs_n": len(matched_group),
            "paired_peak_mean_difference_pp": pd.to_numeric(matched_group.peak_difference_pp).mean(),
            "paired_post_mean_difference_pp": pd.to_numeric(matched_group.post_mean_difference_pp).mean(),
        })
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)


## Inspect an individual burst
Set `BURST_ID` to an index from `events` or the pair table, then rerun this cell. The default is the first matched sustained-reorientation event, not the event with the largest HR response. Window shading marks pre, post, and hold intervals. The angle trace is relative to the pre-window median direction; it is not a body-position trace.


In [ ]:
BURST_ID = None

if events.empty:
    print("No bursts to inspect.")
else:
    if BURST_ID is None:
        BURST_ID = pairs.changed_id.iloc[0] if len(pairs) else events.index[0]
    if BURST_ID not in events.index:
        raise KeyError(f"Unknown BURST_ID: {BURST_ID}")
    event = events.loc[BURST_ID]
    display(event[[
        "start", "end", "AUC", "auc_tertile", "reorientation_class",
        "tilt_change_deg", "post_drift_deg", "pre_rms_g", "post_rms_g", "hold_rms_g",
        "included", "comparison_included", "comparison_exclusion_reason",
    ]].to_frame("value"))
    left = min(event.start - pd.Timedelta(seconds=22), event.pre_start)
    right = max(event.start + pd.Timedelta(seconds=40), event.hold_end + pd.Timedelta(seconds=3))
    raw = axes_g.loc[left:right]
    relative = (raw.index - event.start).total_seconds()
    center = event[["pre_x_g", "pre_y_g", "pre_z_g"]].to_numpy(dtype=float)
    fig, panels = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for column, color in zip(["x_g", "y_g", "z_g"], ["#247b9b", "#b43c69", "#597642"]):
        panels[0].plot(relative, raw[column], color=color, linewidth=0.8, label=column)
    panels[0].set_ylabel("Raw acceleration (g)")
    panels[0].legend(ncol=3, fontsize=8)
    panels[1].plot(relative, angle_degrees(raw.to_numpy(), center), color="#654c94", linewidth=0.8)
    panels[1].set_ylabel("Angle from pre (degrees)")
    epoch = hr["epochs_pct"].loc[BURST_ID]
    panels[2].plot(epoch.index, epoch.to_numpy(dtype=float), color="#303030", label="1 Hz HR epoch")
    callbacks = observations.loc[left:right]
    if np.isfinite(event.baseline_bpm) and event.baseline_bpm > 0:
        panels[2].scatter((callbacks.index - event.start).total_seconds(),
                          (callbacks.hr_bpm / event.baseline_bpm - 1) * 100,
                          s=10, color="#b07827", alpha=0.7, label="Callback-timed HR")
    panels[2].axhline(0, color="0.6", linewidth=0.8)
    panels[2].set(ylabel="HR change (%)", xlabel="Seconds from burst onset")
    panels[2].legend(fontsize=8)
    for ax in panels:
        ax.axvspan(0, (event.end - event.start).total_seconds(), color="#9a768d", alpha=0.13)
        for name, color in [("pre", "#247b9b"), ("post", "#597642"), ("hold", "#b07827")]:
            a = (event[name + "_start"] - event.start).total_seconds()
            b = (event[name + "_end"] - event.start).total_seconds()
            ax.axvspan(a, b, color=color, alpha=0.08)
        ax.axvline(0, color="0.4", linestyle=":", linewidth=0.8)
    panels[0].set_title(f"Burst {BURST_ID}: {event.reorientation_class}; comparison eligible = {event.comparison_included}")
    panels[2].set_xlim((left - event.start).total_seconds(), (right - event.start).total_seconds())
    fig.tight_layout()
    plt.show()


## Optional export
Set `EXPORT = True` in the configuration cell to write event measurements, comparisons, sensitivity results, and settings beneath `outputs/wrist_reorientation/<FIT stem>/`. Re-exporting the same recording replaces these analysis outputs only; the FIT and existing notebook remain untouched.


In [ ]:
settings = {
    "fit_path": str(FIT_PATH.resolve()), "input_unit": INPUT_UNIT,
    "sampling_rate_hz": FS, "burst_threshold_g": BURST_THRESHOLD_G, "merge_gap_s": MERGE_GAP_S,
    "hr_max_gap_s": HR_MAX_GAP_S, "isolation_s": ISOLATION_S,
    "exclude_late_overlap": EXCLUDE_LATE_OVERLAP, "artifact_intervals": ARTIFACTS,
    "orientation": ORIENTATION, "max_auc_ratio": MAX_AUC_RATIO,
    "auc_tertile_cutoffs_g_s": hr["report"]["auc_tertile_cutoffs_g_s"],
    "interpretation": "Exploratory wrist gravity-direction changes, not body sleeping position.",
}
if EXPORT:
    output_dir = PROCESSING / "outputs" / "wrist_reorientation" / FIT_PATH.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, table in {
        "events": events, "group_summary": group_summary, "group_curves": group_curves,
        "matched_pairs": pairs, "pair_summary": pair_summary, "pair_curves": pair_curves,
        "threshold_sensitivity": sensitivity,
    }.items():
        table.to_csv(output_dir / f"{name}.csv", index=True)
    (output_dir / "settings.json").write_text(json.dumps(settings, indent=2, default=str), encoding="utf-8")
    print(f"Exported to {output_dir}")
else:
    print("Export disabled. All tables and figures are available above.")


## What to investigate next
Inspect several examples from every class and review BBI quality around the bursts. Compare matched duration and time separation, then repeat the analysis over multiple nights with a fixed protocol. Any statistical inference should respect nights and participants as sampling units, rather than treating every burst as an independent subject.

Whole-body position would require separate ground truth, such as a synchronized torso sensor or appropriately consented video. This notebook intentionally makes no supine, prone, left-side, or right-side predictions.
